# Tuned lens (Section 8)

**Paper section.** Section 8 (Decoding predictions from intermediate layers) with Figure 8, and Appendix Q with Figures 77 to 94.

**Claim.** In the paper's words, "Residual stream activations carry HMM predictive structure" and "Early-exit predictive accuracy covaries with belief state decodability."

**Experiment.** Per-layer affine lenses trained toward the HMM's ground-truth NTP, toward shuffled and random controls, and toward the model's own output (the canonical lens), scored by the KL to their targets on held-out positions, from `scripts/run_tuned_lens.sh`, compared with the belief probe's R² per layer from `scripts/run_probes.sh`.

**Saved Outputs.** In `figures/`: `tunedlens.pdf`, `tunedlens_all_<model>.pdf`, `tunedlens_corr_slope_<model>.pdf`, and `tuned_concept_corr_<model>.pdf`.

**How the notebook works.** The first code cell sets the paths and loads `results/r2_<model>.csv`. The following cells read `results/tunedlens_<model>.csv`, draw the per-layer KL curves with the belief probe's 1 minus R², compute the per-sequence correlations between the two across layers, and write Figure 8 and the Appendix Q figures to `figures/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
from matplotlib.lines import Line2D
from scipy.stats import pearsonr, spearmanr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.ticker import MultipleLocator

# ═══════════════════════════════════════════════════════════
# CONFIG — change these paths, everything else follows
# ═══════════════════════════════════════════════════════════
RESULTS_DIR = '../results'
PLOT_DIR = '../figures'
os.makedirs(PLOT_DIR, exist_ok=True)

MODEL_KEYS = ['qwen35_9b', 'qwen35_4b', 'llama_31_8b', 'llama_32_3b', 'gemma_4_e4b', 'gemma_4_e2b']
MODEL_LABELS = {
    'qwen35_9b': 'Qwen 3.5 9B', 'qwen35_4b': 'Qwen 3.5 4B',
    'llama_31_8b': 'Llama 3.1 8B', 'llama_32_3b': 'Llama 3.2 3B',
    'gemma_4_e4b': 'Gemma 4 E4B', 'gemma_4_e2b': 'Gemma 4 E2B',
}

# Representative (HMM, param) for single-param plots
TARGETS = [
    ('Mess3', 'a=0.01, x=0.02'),
    # ('Arch', 'a=0.9'),
    ('Arch', 'a=0.99'),
    ('Wing', 'a=0.98, x=0.4'),
    ('Strata', 'a=0.97, t0=0.38, t1=0.54'),
]
HMM_ORDER = ['Mess3', 'Arch', 'Wing', 'Strata']
MODEL_ORDER = list(MODEL_KEYS)
HMM_COLORS = {'Mess3': 'tab:blue', 'Arch': 'tab:orange', 'Wing': 'tab:green', 'Strata': 'tab:red'}
HMMS = ['Mess3', 'Arch', 'Wing', 'Strata']

# ═══════════════════════════════════════════════════════════
# Style
# ═══════════════════════════════════════════════════════════
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'font.weight': 'light',
})
sns.set_context('notebook')
tab10 = sns.color_palette('tab10')

def fmt_tick(v):
    s = f'{v:.1f}' if v == int(v) else f'{v:.2f}'
    if -1 < v < 1 and '.' in s:
        s = s.replace('0.', '.')   # 0.97 → .97  AND  -0.45 → -.45
    return s

# ═══════════════════════════════════════════════════════════
# Loaders
# ═══════════════════════════════════════════════════════════
def load_csv(prefix, model_key):
    path = os.path.join(RESULTS_DIR, f'{prefix}_{model_key}.csv')
    if not os.path.exists(path):
        print(f'  MISSING: {path}')
        return None
    return pd.read_csv(path)

def load_npz(prefix, model_key):
    path = os.path.join(RESULTS_DIR, f'{prefix}_{model_key}.npz')
    if not os.path.exists(path):
        print(f'  MISSING: {path}')
        return None
    return np.load(path, allow_pickle=True)

def harmonize_cross(df):
    if df is None: return None
    if 'source' in df.columns:
        df = df.rename(columns={'source': 'train', 'target': 'test'})
    return df

def harmonize_wing_labels(cross_df, gt_df):
    if cross_df is None or gt_df is None: return cross_df, gt_df
    for name in ['Wing']:
        c_params = set(cross_df[cross_df['hmm']==name]['train'].unique()) if name in cross_df['hmm'].values else set()
        g_params = set(gt_df[gt_df['hmm']==name]['train'].unique()) if name in gt_df['hmm'].values else set()
        if c_params and g_params and c_params != g_params:
            mask = gt_df['hmm'] == name
            for col in ['train', 'test']:
                gt_df.loc[mask, col] = (gt_df.loc[mask, col]
                    .str.replace('x=', 'ALPHA=').str.replace('y=', 'x=').str.replace('ALPHA=', 'a='))
    return cross_df, gt_df

# ═══════════════════════════════════════════════════════════
# Auto-load all R² files
# ═══════════════════════════════════════════════════════════
r2_files = {}
for mk in MODEL_KEYS:
    df = load_csv('r2', mk)
    if df is not None:
        r2_files[mk] = df
        print(f'{MODEL_LABELS[mk]}: {len(df)} rows, HMMs={sorted(df["hmm"].unique())}')
print(f'\n{len(r2_files)} models loaded')

## Figure 8: tuned-lens KL by layer and its correlation with belief decodability (Qwen 3.5 9B)

In [ ]:
# ── Tuned lens: per-lens KL by layer (top) + corr-with-belief bars (bottom) ──
# Controls scored on their own target (kl_self); HMM lens on kl_hmm.
# 1−R² on a REAL right-hand axis (no per-panel rescaling); KL & 1−R² scales fixed
# across all panels — KL labeled on the leftmost, 1−R² on the rightmost.
matplotlib.rcParams.update({'font.family': 'sans-serif',
                            'font.sans-serif': ['DejaVu Sans'], 'font.weight': 'light'})
sns.set_context('notebook')

MODEL = 'qwen35_9b'
SHOW_R2 = True
METRIC = 'pearson'

tl_df = load_csv('tunedlens', MODEL)
r2_df = load_csv('r2', MODEL)

def _klcol(lens):
    return 'kl_hmm' if lens == 'tuned_hmm' else 'kl_self'

_blues = plt.get_cmap('Blues')
_ctrl = [('shuffle', 'TunedLens → Shuffled NTP'), ('random', 'TunedLens → Random NTP')]
_bcols = [_blues(t) for t in np.linspace(0.45, 1.0, len(_ctrl))]
lens_styles = {'tuned_hmm': ('black', 5, '-', 'TunedLens → HMM NTP')}
for (lens, lab), c in zip(_ctrl, _bcols):
    lens_styles[lens] = (c, 5, '-', lab)

_corr_fn = pearsonr if METRIC == 'pearson' else spearmanr
_rng = np.random.default_rng(0)
_bar_lenses = list(lens_styles.keys())
_bar_short = {'tuned_hmm': 'HMM\nNTP', 'shuffle': 'Shuffled\nNTP', 'random': 'Random\nNTP'}

# ---- pre-pass: global fixed limits for KL (left) and 1−R² (right) ----
_kl_all, _omr2_all = [], []
for (hmm, param) in TARGETS:
    d = tl_df[(tl_df['hmm'] == hmm) & (tl_df['param'] == param)]
    for lens in lens_styles:
        m = d[d['lens'] == lens].groupby('layer')[_klcol(lens)].mean()
        if len(m): _kl_all.append(m.values)
    fl = d[(d['lens'] == 'logit') & (d['layer'] == d['layer'].max())]['kl_hmm']
    if len(fl): _kl_all.append(fl.values)
    bel = r2_df[(r2_df['hmm'] == hmm) & (r2_df['param'] == param) &
                (r2_df['target'] == 'real')].groupby('layer')['R2'].mean()
    if len(bel): _omr2_all.append((1 - bel).values)
_kl_all = np.concatenate(_kl_all); _omr2_all = np.concatenate(_omr2_all)
KL_YLIM   = (max(_kl_all[_kl_all > 0].min() * 0.6, 1e-12), _kl_all.max() * 1.6)
_omr2_pos = _omr2_all[_omr2_all > 0]
OMR2_YLIM = (max(_omr2_pos.min() * 0.6, 1e-3), min(_omr2_all.max() * 1.6, 1.0))

fig, axes = plt.subplots(2, 4, figsize=(16, 7.0),
                         gridspec_kw={'height_ratios': [2.5, 2.0]})
_last = len(TARGETS) - 1
_all_corr = []
for i, (hmm, param) in enumerate(TARGETS):
    ax = axes[0, i]
    axb = axes[1, i]
    d = tl_df[(tl_df['hmm'] == hmm) & (tl_df['param'] == param)]

    # ===== TOP-RIGHT axis (twin): 1−R² — create FIRST, push BEHIND so KL grid shows =====
    if SHOW_R2:
        axR = ax.twinx()
        axR.set_yscale('log')
        axR.set_ylim(OMR2_YLIM)
        axR.set_zorder(ax.get_zorder() - 1)     # twin behind the host
        ax.patch.set_visible(False)             # let host grid show through the twin
        axR.patch.set_visible(False)
        rsd = r2_df[(r2_df['hmm'] == hmm) & (r2_df['param'] == param) &
                    (r2_df['target'] == 'real')].groupby('layer')['R2'].agg(['mean', 'sem'])
        mm, sem = rsd['mean'], rsd['sem'].fillna(0)
        om    = np.clip(1 - mm.values, OMR2_YLIM[0], None)
        om_lo = np.clip(1 - (mm + 1.96 * sem).values, OMR2_YLIM[0], None)
        om_hi = np.clip(1 - (mm - 1.96 * sem).values, OMR2_YLIM[0], None)
        axR.plot(mm.index, om, color='red', lw=6, alpha=0.85, zorder=0)
        axR.fill_between(mm.index, om_lo, om_hi, color='red', alpha=0.15, linewidth=0, zorder=0)
        axR.tick_params(axis='y', which='both', right=True, labelright=(i == _last),
                        labelsize=20, length=3, colors='red',
                        labelcolor='red', color='red')
        axR.spines['right'].set_color('red')
        axR.spines['right'].set_linewidth(1.5)
        ax.spines['right'].set_visible(False)
        if i == _last:
            axR.set_ylabel('$1 - R^2$', fontsize=22, color='red')

    # ===== TOP-LEFT axis: per-lens KL by layer (drawn on top of the twin) =====
    for lens, (color, lw, ls, label) in lens_styles.items():
        s = d[d['lens'] == lens]
        if len(s) == 0:
            continue
        stats = s.groupby('layer')[_klcol(lens)].agg(['mean', 'sem']).reset_index()
        band = (1.96 * stats['sem']).fillna(0)
        ax.plot(stats['layer'], stats['mean'], ls=ls, color=color, lw=lw)
        ax.fill_between(stats['layer'], stats['mean'] - band, stats['mean'] + band,
                        color=color, alpha=0.15, linewidth=0)

    final_layer = d['layer'].max()
    fl = d[(d['lens'] == 'logit') & (d['layer'] == final_layer)]['kl_hmm']
    if fl.isna().all() or len(fl) == 0:
        fl = d[(d['lens'] == 'tuned_hmm') & (d['layer'] == final_layer)]['kl_hmm']
    final_kl = fl.mean()
    final_sem = fl.sem() if len(fl) > 1 else 0.0
    final_sem = 0.0 if np.isnan(final_sem) else final_sem
    ax.axhline(final_kl, color='gray', lw=2, ls='-', zorder=1)
    ax.axhspan(final_kl - 1.96 * final_sem, final_kl + 1.96 * final_sem,
               color='gray', alpha=0.15, zorder=0)

    ax.set_yscale('log')
    ax.set_ylim(KL_YLIM)
    ax.set_title(f'{hmm}', fontsize=23, pad=18, y=0.95)
    ax.set_xticks([0, 10, 20, 30])
    ax.set_xlabel('Layer', fontsize=22)
    ax.tick_params(axis='both', labelsize=22, length=3)
    ax.tick_params(axis='y', labelleft=(i == 0))
    ax.grid(True, which='major', alpha=0.2, linewidth=0.5)   # KL-axis grid (now visible)
    ax.grid(False, which='minor')
    if i == 0:
        ax.set_ylabel('Final Token KL', fontsize=22)

    # ===== BOTTOM: corr(lens KL-by-layer, 1 - belief R²) =====
    dpar = tl_df[(tl_df['hmm'] == hmm) & (tl_df['param'] == param)]
    belpar = r2_df[(r2_df['hmm'] == hmm) & (r2_df['param'] == param) & (r2_df['target'] == 'real')]
    per_lens = {l: [] for l in _bar_lenses}
    if ('seed' in dpar.columns) and ('seed' in belpar.columns):
        seeds = sorted(set(dpar['seed'].unique()) & set(belpar['seed'].unique()))
        for sdv in seeds:
            om = 1 - belpar[belpar['seed'] == sdv].groupby('layer')['R2'].mean()
            for lens in _bar_lenses:
                kl = dpar[(dpar['lens'] == lens) & (dpar['seed'] == sdv)].groupby('layer')[_klcol(lens)].mean()
                cm = om.index.intersection(kl.index)
                if len(cm) >= 3:
                    per_lens[lens].append(_corr_fn(kl.loc[cm].values, om.loc[cm].values)[0])
    else:
        om = 1 - belpar.groupby('layer')['R2'].mean()
        for lens in _bar_lenses:
            kl = dpar[dpar['lens'] == lens].groupby('layer')[_klcol(lens)].mean()
            cm = om.index.intersection(kl.index)
            if len(cm) >= 3:
                per_lens[lens].append(_corr_fn(kl.loc[cm].values, om.loc[cm].values)[0])
    _bar_df = pd.DataFrame(
        [(l, v) for l in _bar_lenses for v in per_lens[l] if np.isfinite(v)],
        columns=['lens', 'corr'])
    cols = [lens_styles[l][0] for l in _bar_lenses]
    sns.barplot(data=_bar_df, x='lens', y='corr', order=_bar_lenses, hue='lens',
                hue_order=_bar_lenses, palette=cols, legend=False,
                errorbar=('ci', 95), ax=axb)
    _all_corr.extend(_bar_df['corr'].tolist())
    axb.axhline(0, color='black', lw=1)
    axb.set_xlabel('')
    axb.set_ylabel('corr(KL, 1−R²)' if i == 0 else '', fontsize=22)
    axb.set_xticks(np.arange(len(_bar_lenses)))
    axb.set_xticklabels([_bar_short[l] for l in _bar_lenses], fontsize=15)
    axb.tick_params(axis='y', labelsize=16, length=3)
    axb.grid(True, axis='y', alpha=0.2, linewidth=0.5)
    axb.set_axisbelow(True)

for axb in axes[1]:
    axb.set_ylim(-0.65, 1.0)
    axb.set_yticks([-0.5, 0.0, 0.5, 1.0])

# single legend: row1 = HMM (black), Shuffled, Random; row2 = 1−R² (red), Baseline (gray)
_blk = lens_styles['tuned_hmm']
_handles = [Line2D([], [], color=_blk[0], lw=5, ls=_blk[2], label=_blk[3])]   # black HMM
_handles += [Line2D([], [], color=c, lw=5, ls=ls, label=lab)                   # shuffled, random
             for lens, (c, lw, ls, lab) in lens_styles.items() if lens != 'tuned_hmm']
if SHOW_R2:
    _handles.append(Line2D([], [], color='red', lw=5, label='1 − Belief Probe R²'))  # red


_handles = [
    Line2D([], [], color=lens_styles['tuned_hmm'][0], lw=5,
           ls=lens_styles['tuned_hmm'][2], label='TunedLens → Ground-Truth NTP'),
    Line2D([], [], color='red', lw=5, label='1 − Belief Probe R²'),
    Line2D([], [], color=lens_styles['shuffle'][0], lw=5,
           ls=lens_styles['shuffle'][2], label='TunedLens → Shuffled NTP'),
    Line2D([], [], color='0.5', lw=5, ls='-', label='Original KL of LLM'),
    Line2D([], [], color=lens_styles['random'][0], lw=5,
           ls=lens_styles['random'][2], label='TunedLens → Random NTP'),
]

plt.tight_layout(w_pad=0, h_pad=1.0)
fig.legend(handles=_handles, loc='lower center', ncol=3,
           fontsize=22, frameon=False, bbox_to_anchor=(0.5, -0.175),
           columnspacing=1.4, handletextpad=0.5, handlelength=1.5)
plt.savefig(f'{PLOT_DIR}/tunedlens.pdf', bbox_inches='tight')
plt.show(); plt.close()

## Appendix Q.2: per-layer KL for every parametrization (Figures 83 to 88)

In [ ]:
# ── Tuned lens (appendix): ALL params overlapped per family panel, 2x2 per model ──
# Per panel (one family): for each parametrization —
#   - HMM-lens KL (kl_hmm), viridis gradient, prominent (host axis, log)
#   - shuffled / random control KL (kl_self), light/dark blue, faint thin
#   - 1 − belief-probe R², Reds gradient, on a red twin axis (log), behind the host
# Simplified annotation like the r2_controls cell: per-param colors come from the
# main (viridis) curve; controls and 1−R² get generic legend entries.
for model_key in MODEL_KEYS:
    model_name = MODEL_LABELS.get(model_key, model_key)
    tl_df = load_csv('tunedlens', model_key)
    r2_df_model = load_csv('r2', model_key)
    if tl_df is None or r2_df_model is None:
        continue
    tl_df = tl_df[tl_df['hmm'] != 'Spiral']
    r2_df_model = r2_df_model[r2_df_model['hmm'] != 'Spiral']
    hmms = [h for h in HMM_ORDER if h in tl_df['hmm'].values]

    def _klcol(lens):
        return 'kl_hmm' if lens == 'tuned_hmm' else 'kl_self'

    # global fixed limits across the model (KL host, 1-R^2 twin)
    _kl_all, _om_all = [], []
    for hmm in hmms:
        for param in sorted(tl_df[tl_df['hmm'] == hmm]['param'].unique()):
            d = tl_df[(tl_df['hmm'] == hmm) & (tl_df['param'] == param)]
            for lens in ['tuned_hmm', 'shuffle', 'random']:
                m = d[d['lens'] == lens].groupby('layer')[_klcol(lens)].mean()
                if len(m): _kl_all.append(m.values)
            bel = r2_df_model[(r2_df_model['hmm'] == hmm) & (r2_df_model['param'] == param) &
                              (r2_df_model['target'] == 'real')].groupby('layer')['R2'].mean()
            if len(bel): _om_all.append((1 - bel).values)
    _kl_all = np.concatenate(_kl_all); _om_all = np.concatenate(_om_all)
    KL_YLIM = (max(_kl_all[_kl_all > 0].min() * 0.6, 1e-12), _kl_all.max() * 1.6)
    _omp = _om_all[_om_all > 0]
    OM_YLIM = (max(_omp.min() * 0.6, 1e-3), min(_om_all.max() * 1.6, 1.0))

    fig = plt.figure(figsize=(13, 8))
    fig.patch.set_facecolor('white'); fig.patch.set_edgecolor('white'); fig.patch.set_linewidth(0)
    outer = fig.add_gridspec(2, 2, wspace=0.55, hspace=0.3)
    axes_list = [fig.add_subplot(outer[r, c]) for r in range(2) for c in range(2)]

    C_SHUF, C_RAND = '#9ecae1', '#08519c'   # light / dark blue (match main figure)

    for j, hmm in enumerate(hmms[:4]):
        ax = axes_list[j]
        r, c = divmod(j, 2)
        _right_col = (c == 1)
        hmm_tl = tl_df[tl_df['hmm'] == hmm]
        params = sorted(hmm_tl['param'].unique())
        tp = np.linspace(0.3, 0.9, len(params))
        colors_main = plt.cm.viridis(tp)
        colors_om   = plt.cm.Reds(tp)

        # twin FIRST, pushed behind, so the host grid shows
        axR = ax.twinx()
        axR.set_yscale('log'); axR.set_ylim(OM_YLIM)
        axR.set_zorder(ax.get_zorder() - 1)
        ax.patch.set_visible(False); axR.patch.set_visible(False)

        for i, param in enumerate(params):
            d = hmm_tl[hmm_tl['param'] == param]
            # controls: faint thin
            for lens, col in [('shuffle', C_SHUF), ('random', C_RAND)]:
                s = d[d['lens'] == lens]
                if len(s):
                    stats = s.groupby('layer')[_klcol(lens)].agg(['mean']).reset_index()
                    ax.plot(stats['layer'], stats['mean'], '-', color=col, lw=0.6, alpha=0.5)
            # HMM lens: prominent viridis
            s = d[d['lens'] == 'tuned_hmm']
            if len(s):
                stats = s.groupby('layer')['kl_hmm'].agg(['mean', 'sem']).reset_index()
                band = (1.96 * stats['sem']).fillna(0)
                ax.plot(stats['layer'], stats['mean'], '-', color=colors_main[i], lw=1.5)
                ax.fill_between(stats['layer'], stats['mean'] - band, stats['mean'] + band,
                                color=colors_main[i], alpha=0.12, linewidth=0)
            # 1 - R^2 on the twin: Reds gradient
            bel = r2_df_model[(r2_df_model['hmm'] == hmm) & (r2_df_model['param'] == param) &
                              (r2_df_model['target'] == 'real')].groupby('layer')['R2'].mean()
            if len(bel):
                om = np.clip(1 - bel.values, OM_YLIM[0], None)
                axR.plot(bel.index, om, '-', color=colors_om[i], lw=1.0, alpha=0.8, zorder=0)

        ax.set_yscale('log'); ax.set_ylim(KL_YLIM)
        ax.set_title(hmm, fontsize=16)
        n_layers = int(hmm_tl['layer'].max())
        ax.set_xlim(-0.5, n_layers + 0.5)
        ax.set_xticks([0, 10, 20, 30])
        ax.tick_params(axis='both', labelsize=13, length=3)
        ax.grid(True, which='both', alpha=0.2, linewidth=0.5)
        if r == 1:
            ax.set_xlabel('Layer', fontsize=14)

        # red right axis; labels on right-column panels only
        axR.tick_params(axis='y', which='both', right=True, labelright=_right_col,
                        labelsize=11, length=3, colors='red',
                        labelcolor='red', color='red')
        axR.spines['right'].set_color('red'); axR.spines['right'].set_linewidth(1.2)
        ax.spines['right'].set_visible(False)
        if _right_col:
            axR.set_ylabel('$1 - R^2$', fontsize=13, color='red')

        # legend in the wspace gap; pushed further right where red labels exist
        param_handles = [Line2D([], [], color=colors_main[i], lw=3, label=p)
                         for i, p in enumerate(params)]
        param_handles.append(Line2D([], [], color=C_SHUF, lw=3, label='Shuffled NTP'))
        param_handles.append(Line2D([], [], color=C_RAND, lw=3, label='Random NTP'))
        param_handles.append(Line2D([], [], color='red', lw=3, label='1 − R²'))
        ax.legend(handles=param_handles, fontsize=9.5, ncol=1, loc='center left',
                  bbox_to_anchor=(1.16 if _right_col else 1.02, 0.5),
                  frameon=True, edgecolor='0.8',
                  borderpad=0.4, handlelength=1.2, labelspacing=0.3)

    # ylabel on the two left-column panels
    fig.canvas.draw()
    for ax in (axes_list[0], axes_list[2]):
        pos = ax.get_position()
        fig.text(pos.x0 - 0.05, (pos.y0 + pos.y1) / 2,
                 'Final Token KL', fontsize=15, va='center', ha='center', rotation=90)

    fig.suptitle(model_name, fontsize=16, y=0.98)
    plt.savefig(f'{PLOT_DIR}/tunedlens_all_{model_key}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

## Appendix Q.3: correlations for every parametrization (Figures 89 to 94)

In [ ]:
# Slopegraph: one line per parametrization across the lenses (x), correlation on y.
# Per-param value = corr per seed -> mean over that param's seeds.
# METRIC (per Xavier): each lens uses its OWN-target KL — HMM lens -> kl_hmm,
# controls -> kl_self — matching the main tuned-lens figure.
from matplotlib.lines import Line2D

_slope_lenses = ['tuned_hmm', 'shuffle', 'random']
_slope_short  = {'tuned_hmm': 'HMM\nNTP', 'shuffle': 'Shuffled\nNTP', 'random': 'Random\nNTP'}
def _slope_klcol(lens):
    return 'kl_hmm' if lens == 'tuned_hmm' else 'kl_self'

for model_key in MODEL_KEYS:
    model_name = MODEL_LABELS.get(model_key, model_key)
    tl_df = load_csv('tunedlens', model_key)
    r2_df = load_csv('r2', model_key)
    if tl_df is None or r2_df is None:
        continue

    hmms = [h for h in HMM_ORDER if h in tl_df['hmm'].values]
    fig, axes = plt.subplots(1, len(hmms), figsize=(3.4 * len(hmms), 5), squeeze=False)
    axes = axes[0]
    x = np.arange(len(_slope_lenses))

    for ai, hmm in enumerate(hmms):
        ax = axes[ai]
        params = sorted(tl_df[tl_df['hmm'] == hmm]['param'].unique())
        pcols = plt.get_cmap('viridis')(np.linspace(0.15, 0.9, len(params)))

        col_vals = {li: [] for li in range(len(_slope_lenses))}
        for pi, pp in enumerate(params):
            dpar   = tl_df[(tl_df['hmm'] == hmm) & (tl_df['param'] == pp)]
            belpar = r2_df[(r2_df['hmm'] == hmm) & (r2_df['param'] == pp) &
                           (r2_df['target'] == 'real')]
            seeds = sorted(set(dpar['seed'].unique()) & set(belpar['seed'].unique()))
            y = np.full(len(_slope_lenses), np.nan)
            for li, lens in enumerate(_slope_lenses):
                col = _slope_klcol(lens)
                cs = []
                for sdv in seeds:
                    om = 1 - belpar[belpar['seed'] == sdv].groupby('layer')['R2'].mean()
                    kl = dpar[(dpar['lens'] == lens) & (dpar['seed'] == sdv)] \
                            .groupby('layer')[col].mean()
                    cm = om.index.intersection(kl.index)
                    if len(cm) >= 3:
                        cs.append(_corr_fn(kl.loc[cm].values, om.loc[cm].values)[0])
                if cs:
                    y[li] = np.mean(cs); col_vals[li].append(y[li])
            ax.plot(x, y, '-', color=pcols[pi], lw=1.3, alpha=0.7,
                    marker='o', ms=3, zorder=2, label=pp)

        ymean = [np.nanmean(col_vals[li]) if col_vals[li] else np.nan
                 for li in range(len(_slope_lenses))]
        ax.plot(x, ymean, '-', color='black', lw=3, marker='o', ms=6, zorder=3, label='mean')

        ax.axhline(0, color='0.6', lw=1, zorder=0)
        ax.set_xticks(x)
        ax.set_xticklabels([_slope_short[l] for l in _slope_lenses], fontsize=11)
        ax.set_xlim(-0.3, len(_slope_lenses) - 0.7)
        ax.set_ylim(-0.65, 1.0); ax.set_yticks([-0.5, 0.0, 0.5, 1.0])
        ax.tick_params(axis='y', labelsize=13, length=3)
        ax.set_title(hmm, fontsize=15)
        ax.grid(True, axis='y', alpha=0.2, linewidth=0.5); ax.set_axisbelow(True)
        if ai == 0:
            ax.set_ylabel('corr(KL, 1−R²)', fontsize=14)

        handles = [Line2D([], [], color=pcols[pi], lw=2, marker='o', ms=3, label=str(pp))
                   for pi, pp in enumerate(params)]
        handles.append(Line2D([], [], color='black', lw=3, marker='o', ms=5, label='mean'))
        ncol = 2 if len(handles) > 6 else 1
        ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.3),
                  ncol=ncol, fontsize=7.5, frameon=False, handlelength=1.4,
                  columnspacing=1.0, labelspacing=0.3)

    fig.suptitle(model_name, fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/tunedlens_corr_slope_{model_key}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

## Appendix Q.1: the canonical lens (Figures 77 to 82, bottom panels)

In [ ]:
# ══ canonical tuned lens: corr(per-layer KL, 1 − belief R²), per-seed, all models ══
#   For each (model, hmm, param, seed): pearson r between the lens's KL-by-layer curve
#   and 1 − belief-probe R² by layer. tuned_concept scored on kl_final (its objective,
#   KL to the model's final-layer distribution) and on kl_hmm (KL to ground truth);
#   tuned_hmm (kl_hmm) shown as reference. Reported: mean r over seeds×params per family.
_CLENSES = [('tuned_concept', 'kl_final'), ('tuned_concept', 'kl_hmm'),
            ('tuned_hmm',     'kl_hmm')]

rows = []
for mk in MODEL_ORDER:
    tl_df = load_csv('tunedlens', mk)
    r2_df = load_csv('r2', mk)
    if tl_df is None or r2_df is None:
        continue
    tl_df = tl_df[tl_df.hmm != 'Spiral']
    for (hmm, param), dpar in tl_df.groupby(['hmm', 'param']):
        belpar = r2_df[(r2_df.hmm == hmm) & (r2_df.param == param) &
                       (r2_df.target == 'real')]
        seeds = sorted(set(dpar.seed.unique()) & set(belpar.seed.unique()))
        for sdv in seeds:
            om = 1 - belpar[belpar.seed == sdv].groupby('layer')['R2'].mean()
            for lens, klcol in _CLENSES:
                kl = dpar[(dpar.lens == lens) & (dpar.seed == sdv)] \
                        .groupby('layer')[klcol].mean()
                cm = om.index.intersection(kl.index)
                if len(cm) >= 3:
                    r, _ = pearsonr(kl.loc[cm].values, om.loc[cm].values)
                    rows.append(dict(model=mk, hmm=hmm, param=param, seed=sdv,
                                     lens=lens, klcol=klcol, r=r))
corr_df = pd.DataFrame(rows)

print('mean per-seed pearson r  (rows: model × family; columns: lens/score)')
piv = corr_df.pivot_table(index=['model', 'hmm'], columns=['lens', 'klcol'],
                          values='r', aggfunc='mean').round(2)
print(piv.to_string())
print()
print('grand summary (mean ± sd of per-seed r, pooled over models, families, params, seeds)')
for lens, klcol in _CLENSES:
    s = corr_df[(corr_df.lens == lens) & (corr_df.klcol == klcol)]['r']
    print(f"  {lens:14s} {klcol:9s}  mean r = {s.mean():+.2f} ± {s.std():.2f}   "
          f"%r>0: {100*(s>0).mean():.0f}%   n={len(s)}")

In [ ]:
# ══ canonical tuned lens: corr(per-layer KL, 1 − belief R²) — bars per model, 1×4 ══
#   Bars: mean per-seed pearson r with 95% CI (n = 10 params × 10 seeds per family).
_BAR_SPECS = [('tuned_concept', 'kl_final', 'Canonical\n(KL to final)', '0.25'),
              ('tuned_hmm',     'kl_hmm',   'HMM lens\n(KL to truth)',  'black')]

for mk in MODEL_ORDER:
    sm = corr_df[corr_df.model == mk]
    if len(sm) == 0: continue
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
    print(f"{MODEL_LABELS.get(mk, mk)}")
    for i, (ax, hmm) in enumerate(zip(axes, HMMS)):
        for xi, (lens, klcol, lab, col) in enumerate(_BAR_SPECS):
            s = sm[(sm.hmm == hmm) & (sm.lens == lens) & (sm.klcol == klcol)]['r'].dropna()
            if len(s) == 0: continue
            m = s.mean(); ci = 1.96 * s.std(ddof=1) / np.sqrt(len(s))
            ax.bar(xi, m, color=col, width=0.65)
            ax.errorbar(xi, m, yerr=ci, color='red' if col == 'black' else 'k',
                        lw=1.8, capsize=4, zorder=3)
            print(f"    {hmm:7s} {lens:14s} {klcol:9s} r = {m:+.2f} ± {ci:.2f}  n={len(s)}")
        ax.axhline(0, color='black', lw=1)
        ax.set_ylim(-1, 1)
        ax.set_xticks(range(len(_BAR_SPECS)))
        ax.set_xticklabels([lab for *_ , lab, _c in _BAR_SPECS], fontsize=11)
        ax.set_title(hmm, fontsize=20)
        ax.tick_params(axis='y', labelsize=14, length=3)
        ax.grid(True, axis='y', alpha=0.2, linewidth=0.5); ax.set_axisbelow(True)
        if i == 0:
            ax.set_ylabel('corr(KL, $1-R^2$)', fontsize=15)
    fig.suptitle(MODEL_LABELS.get(mk, mk), fontsize=17, y=1.02)
    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/tuned_concept_corr_{mk}.pdf', bbox_inches='tight')
    plt.show(); plt.close()